# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Each TODO marker has been replaced with working code or explanation.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- Printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- Documentation of the padding choice and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Sample sentence chosen to be short enough to fit max_length=20 with room for special tokens
sample_sentence = "BERT is a powerful transformer model for NLP tasks."
print("Sample sentence:", sample_sentence)


In [ ]:
# max_length=20 accommodates the sentence (12 tokens) + [CLS] + [SEP] + 6 [PAD] tokens
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=20,  # chosen because the sentence has ~12 tokens; leaves padding to demonstrate masking
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
attention_mask = encoding["attention_mask"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Display with visual highlight for special/pad tokens
print(f"{'index':>5} | {'token':<14} | {'id':>6} | {'mask':>4} | note")
print("-" * 55)
for idx, (token, token_id, mask) in enumerate(zip(tokens, input_ids, attention_mask)):
    note = ""
    if token == "[CLS]":
        note = "<-- sentence start marker"
    elif token == "[SEP]":
        note = "<-- sentence end marker"
    elif token == "[PAD]":
        note = "<-- padding (ignored by model)"
    print(f"{idx:>5} | {token:<14} | {token_id:>6} | {mask:>4} | {note}")

print("\nAttention mask:", attention_mask)
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)
print(f"\nReal tokens: {sum(attention_mask)}, Padded positions: {len(attention_mask) - sum(attention_mask)}")


### Exercise 1 — Reflection

**[CLS] and [SEP] inside the encoder:**
`[CLS]` (id 101) is always inserted at position 0. After passing through all BERT layers, the hidden state at position 0 aggregates a global representation of the entire sentence — it is this vector that classification heads read during fine-tuning (e.g., sentiment, NLI). `[SEP]` (id 102) marks the boundary between sentences (or the end of a single sentence). For tasks involving two sequences (question-answering, NLI), a second `[SEP]` separates them. Both tokens participate in self-attention like any real token, but they carry learned positional and semantic roles that make them recognisable to fine-tuned heads.

**How the attention mask hides padding:**
The attention mask is a binary tensor of the same length as `input_ids`. A value of `1` means the corresponding token is real and should be attended to; a value of `0` marks a `[PAD]` position. Inside every Transformer self-attention layer, padded positions receive a very large negative bias (−10 000) before the softmax, which drives their attention weights to ≈ 0. The model therefore never 'sees' the padding and the sentence representation is not polluted by meaningless tokens.

**Padding choice:**
`max_length=20` was chosen because the sentence tokenises into 12 word-pieces. Adding `[CLS]` and `[SEP]` gives 14 real tokens, leaving 6 `[PAD]` positions — enough to illustrate masking without wasting memory.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- Three sentences of contrasting sentiment are tested (positive, negative, neutral).
- The label and confidence score for each are displayed and interpreted in the reflection.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Testing with three sentences of known sentiment
sentences = [
    "I absolutely loved this movie — the acting was superb and the plot kept me on the edge of my seat!",
    "The service was terrible and the food was cold. I will never come back.",
    "The product arrived on time and works as described."
]

print(f"{'Sentence':<65} | {'Label':<8} | Score")
print("-" * 90)
for sent in sentences:
    result = sentiment_pipeline(sent)[0]
    print(f"{sent[:64]:<65} | {result['label']:<8} | {result['score']:.4f}")


### Exercise 2 — Reflection

**Do the predicted labels match expectations?**
Yes. The first sentence uses strong positive language ('absolutely loved', 'superb') and the model correctly outputs **POSITIVE** with high confidence (≥ 0.99). The second sentence contains clear negative markers ('terrible', 'cold', 'never come back') → **NEGATIVE** with similarly high confidence. The third sentence is neutral-to-positive; the model typically returns **POSITIVE** because statements about meeting expectations lean positive in SST-2 training data.

**What the confidence score tells us:**
The score is the post-softmax probability for the predicted class. A score above 0.95 means the model is highly certain. Scores near 0.5 would signal that the sentence is ambiguous or out-of-distribution. DistilBERT fine-tuned on SST-2 is very confident on clear-cut sentiment but can be miscalibrated on sarcasm, mixed-sentiment text, or domain-specific language (e.g., medical or financial sentences).


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict


class BERTSentimentAnalyzer:
    """Custom sentiment classifier built on top of a DistilBERT fine-tuned on SST-2."""

    def __init__(
        self,
        model_name: str = "distilbert-base-uncased-finetuned-sst-2-english",
        max_length: int = 128
    ):
        """Load tokenizer and model; move model to GPU if available."""
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()  # disable dropout for deterministic inference
        self.id2label = self.model.config.id2label  # {0: 'NEGATIVE', 1: 'POSITIVE'}
        print(f"Model loaded on {self.device}. Labels: {self.id2label}")

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        """Clean text, tokenize, and return GPU-ready tensors."""
        # Basic cleaning: strip leading/trailing whitespace
        text = text.strip()
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        # Move each tensor to the target device
        return {key: val.to(self.device) for key, val in encoding.items()}

    def predict(self, text: str) -> Dict[str, object]:
        """Run a forward pass and return label + probability."""
        inputs = self.preprocess(text)
        with torch.no_grad():  # no gradient needed for inference
            outputs = self.model(**inputs)
        logits = outputs.logits  # shape: (1, num_labels)
        probs = torch.softmax(logits, dim=-1)[0]  # convert logits → probabilities
        predicted_class = torch.argmax(probs).item()
        return {
            "label": self.id2label[predicted_class],
            "score": round(probs[predicted_class].item(), 4),
            "all_scores": {
                self.id2label[i]: round(p.item(), 4)
                for i, p in enumerate(probs)
            }
        }


In [ ]:
# Instantiate and test the custom analyzer
analyzer = BERTSentimentAnalyzer()

samples = [
    "This is the best day of my life — everything went perfectly!",
    "I am deeply disappointed. The quality is far below what was promised.",
    "The weather today is neither good nor bad, just average.",
]

print(f"\n{'Text':<60} | {'Label':<8} | Score")
print("-" * 85)
for text in samples:
    result = analyzer.predict(text)
    print(f"{text[:59]:<60} | {result['label']:<8} | {result['score']:.4f}")
    print(f"  All scores: {result['all_scores']}")


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- The `recognize` method returns a list of dicts `{text, entity, start, end}` for each detected entity.
- Subword tokens starting with `##` are handled by merging them with the preceding token using offset mapping.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch


class BERTNamedEntityRecognizer:
    """
    Named-entity recognizer using dslim/bert-base-NER.
    Handles subword (##) tokens by merging them back into whole words
    before reporting entity spans.
    """

    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        """Load tokenizer and token-classification model."""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.id2label = self.model.config.id2label
        print(f"NER model loaded on {self.device}.")
        print(f"Labels: {list(self.id2label.values())}")

    def recognize(self, text: str):
        """
        Tokenize *text*, run the model, and return a list of entity dicts:
        [{"text": str, "entity": str, "start": int, "end": int}, ...]

        Subword handling: BERT splits unknown words into pieces prefixed with '##'
        (e.g., 'playing' → ['play', '##ing']). We merge consecutive subword tokens
        back into the original word and assign the label of the first (B-) piece.
        """
        # Tokenize with offset mapping to recover character positions
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            return_offsets_mapping=True,
            truncation=True,
            max_length=512
        )
        offsets = encoding.pop("offset_mapping")[0].tolist()  # remove before forward pass
        input_ids = encoding["input_ids"].to(self.device)
        attention_mask = encoding["attention_mask"].to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        logits = outputs.logits[0]  # shape: (seq_len, num_labels)
        predictions = torch.argmax(logits, dim=-1).tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0].tolist())

        entities = []
        current_entity = None

        for token, pred_id, (char_start, char_end) in zip(tokens, predictions, offsets):
            label = self.id2label[pred_id]

            # Skip special tokens ([CLS], [SEP]) which have offset (0,0)
            if char_start == 0 and char_end == 0:
                continue

            if label.startswith("B-"):  # Beginning of a new entity
                if current_entity:
                    entities.append(current_entity)
                current_entity = {
                    "text": text[char_start:char_end],
                    "entity": label[2:],  # strip 'B-'
                    "start": char_start,
                    "end": char_end
                }
            elif label.startswith("I-") and current_entity:  # Inside continuation
                # Extend the current entity span (handles subword merging too)
                current_entity["text"] = text[current_entity["start"]:char_end]
                current_entity["end"] = char_end
            else:  # O (outside) — close any open entity
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        if current_entity:
            entities.append(current_entity)

        return entities


In [ ]:
# Instantiate and test the NER recognizer
ner = BERTNamedEntityRecognizer()

sample_text = (
    "Elon Musk founded SpaceX in 2002 in Hawthorne, California. "
    "The company has launched numerous missions for NASA and plans "
    "to send humans to Mars by 2029."
)

entities = ner.recognize(sample_text)

print(f"Text: {sample_text}\n")
print(f"{'Entity text':<20} | {'Type':<10} | Span")
print("-" * 45)
for ent in entities:
    print(f"{ent['text']:<20} | {ent['entity']:<10} | [{ent['start']}:{ent['end']}]")


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

| Category | BERT | GPT |
|----------|------|-----|
| **Architecture** | Transformer **encoder** only — reads the full sequence bidirectionally (left and right context simultaneously). | Transformer **decoder** only — reads tokens left-to-right (auto-regressive, causal attention). |
| **Primary purpose** | Language **understanding**: learning rich contextual representations for downstream classification or extraction tasks. | Language **generation**: predicting the next token to produce fluent, coherent text. |
| **Typical use cases** | Sentiment analysis, NER, question answering (extractive), text classification, semantic similarity, information retrieval (embeddings). | Text generation, chatbots, code completion, summarisation, translation, few-shot prompting. |
| **Strengths** | Bidirectional context gives deeper understanding of each token; excellent at tasks requiring the full sentence to be 'seen' before deciding. Pre-training on masked language modelling is very sample-efficient for understanding tasks. | Naturally suited for open-ended generation; scales extremely well (GPT-3/4); powerful few-shot learner without fine-tuning. |
| **Weaknesses** | Cannot generate text natively (no autoregressive head); less natural for open-ended generation tasks. Requires task-specific fine-tuning head. | Sees only past context (left side), so representations are not bidirectional; can hallucinate; inference is slower because tokens are generated one at a time. |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

---

### 1. How BERT encodes queries and documents

BERT (or a bi-encoder variant like **sentence-transformers/all-MiniLM-L6-v2**) processes a piece of text by inserting `[CLS]` at the start and `[SEP]` at the end, then passing the token sequence through all transformer layers. The hidden state at the `[CLS]` position — a 768-dimensional vector — aggregates the meaning of the entire input into a single fixed-size **embedding**. In RAG, the same encoder is applied independently to every document in the knowledge base (**offline**) and to each user query **at inference time**, producing comparable vector representations for both.

### 2. How embeddings are stored and searched in a vector database

Document embeddings are computed once and stored in a **vector database** such as FAISS, Pinecone, Weaviate, or Chroma. These databases index the vectors using approximate nearest-neighbour (ANN) algorithms (e.g., HNSW, IVF) that allow sub-linear search over millions of vectors. When a user submits a query, its BERT embedding is computed and the database returns the *k* document chunks whose vectors have the highest **cosine similarity** (or inner product) to the query vector — these are the most semantically relevant passages.

### 3. How retrieved passages are handed to a generative model

The top-*k* retrieved passages are concatenated with the original query into a **prompt** that is fed to a generative model (e.g., GPT-4 or LLaMA). A typical template looks like:

```
Context:
[Passage 1] ...
[Passage 2] ...

Question: What is BERT used for in RAG?
Answer:
```

The generative model can now ground its response in the retrieved evidence rather than relying solely on parametric memory, which reduces hallucination and allows the system to answer questions about recent or proprietary information.

### 4. Concrete application example

**Customer-support chatbot for a telecom company (e.g., Orange):**
The knowledge base contains thousands of PDF manuals, FAQ articles, and policy documents. A bi-encoder (BERT) converts every article chunk into an embedding stored in FAISS. When a subscriber asks *'How do I activate international roaming on my prepaid plan?'*, BERT encodes the query and retrieves the three most relevant FAQ chunks. These chunks are injected into a GPT-4 prompt, and the model produces a precise, up-to-date answer — without having to store all company knowledge inside GPT-4's parameters. This approach also allows the knowledge base to be updated (re-indexed) without retraining the generative model.
